[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Smiledxd/python_data/blob/main/sesiones/S25_sql_desde_python.ipynb)

# Sesión 25 · SQL desde Python

**Módulo 6: Negocio y extras** · ⏱️ Duración estimada: 60 a 75 minutos

## 🎯 Objetivos
Al terminar esta sesión podrás:
1. Conectarte a una base SQLite con `sqlite3` y traer resultados a pandas con `pd.read_sql`.
2. Filtrar y ordenar con `WHERE` y `ORDER BY`, usando parámetros de forma segura.
3. Combinar tablas con `JOIN` y `LEFT JOIN`.
4. Resumir con `GROUP BY` y filtrar grupos con `HAVING`.
5. Organizar consultas largas con CTE (`WITH`) y usar funciones de ventana (`OVER`).

## 📋 Qué debes saber antes
Módulo 3 (pandas): filtrar, `groupby` y `merge`. Hoy harás lo mismo, pero en SQL.

## 🧭 Cómo trabajar este notebook
- Ejecuta las celdas **en orden**, de arriba abajo, con **Shift + Enter**.
- En cada ✍️ **Tu turno** escribe tu consulta como texto (entre triples comillas) y ejecútala con `pd.read_sql`.
- La celda ✅ **Verificar** corre tu consulta en la base del curso **y** en una base de prueba pequeña con casos borde: tu SQL tiene que funcionar en las dos.
- Si te atascas, abre la 💡 **Pista**. Hay dos, de menos a más ayuda.

## ⚙️ Setup
Ejecuta la celda siguiente **al empezar** (y otra vez si reinicias el entorno). Crea la base de datos en memoria, aplica el estilo de gráficos y carga las funciones que revisan tus respuestas.

⚠️ **Ejecútala y no la edites.**

In [ ]:
#@title ⚙️ Setup: ejecuta esta celda y no la edites { display-mode: "form" }
# Crea la base de datos de la sesión, aplica el estilo de gráficos y carga los verificadores.
import copy
import hashlib
import math
import sqlite3
import statistics

import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# ---------- Estilo de los gráficos ----------
# Paleta categórica en orden fijo (validada para daltonismo) y tintas para textos y ejes.
AZUL, NARANJA, AQUA, AMARILLO, MAGENTA, VERDE, VIOLETA, ROJO = (
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948")
GRIS = "#c3c2b7"          # para lo que no es protagonista
TINTA = "#0b0b0b"         # textos principales
TINTA_2 = "#52514e"       # textos secundarios
FONDO = "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO, "savefig.facecolor": FONDO,
    "axes.edgecolor": GRIS, "axes.labelcolor": TINTA_2, "text.color": TINTA,
    "xtick.color": "#898781", "ytick.color": "#898781",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e1e0d9", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.prop_cycle": plt.cycler(color=[AZUL, NARANJA, AQUA, AMARILLO, MAGENTA, VERDE, VIOLETA, ROJO]),
    "axes.titlesize": 13, "axes.titlelocation": "left", "axes.titleweight": "bold",
    "lines.linewidth": 2, "font.size": 11, "figure.dpi": 100,
    "axes.formatter.useoffset": False, "axes.formatter.limits": (-9, 9),   # sin notación científica (1e6)
})



# ---------- Datos de práctica: ventas de una cadena de tiendas, enero a junio de 2025 ----------
_tiendas = pd.DataFrame({
    "id_tienda": range(1, 9),
    "nombre": ["Tienda Centro", "Tienda Miraflores", "Tienda Trujillo", "Tienda Piura", "Tienda Arequipa", "Tienda Cusco", "Tienda Surco", "Tienda Chiclayo"],
    "region": ["Lima", "Lima", "Norte", "Norte", "Sur", "Sur", "Lima", "Norte"],
    "apertura": ["2019-03-01", "2020-07-15", "2018-11-01", "2021-02-01", "2019-09-01", "2022-04-01", "2023-01-10", "2025-07-01"],
})
_cats = {"Bebidas": [("Agua 625 ml", 1.8), ("Gaseosa 1.5 L", 6.5), ("Jugo de naranja 1 L", 7.9), ("Café premium 250 g", 32.0)],
         "Snacks": [("Papas fritas 150 g", 5.4), ("Galletas de avena", 3.2), ("Maní salado 200 g", 6.9)],
         "Lácteos": [("Leche entera 1 L", 4.6), ("Yogur natural 1 L", 8.3), ("Queso fresco 500 g", 14.9)],
         "Limpieza": [("Detergente 900 g", 12.5), ("Lejía 1 L", 4.2), ("Jabón líquido 500 ml", 9.9), ("Papel higiénico x4", 8.7), ("Esponjas x3", 3.5)]}
_productos = pd.DataFrame([(n, c, p) for c, ps in _cats.items() for n, p in ps], columns=["nombre", "categoria", "precio"])
_productos.insert(0, "id_producto", range(1, len(_productos) + 1))
_vendibles = _productos["id_producto"][_productos["nombre"] != "Café premium 250 g"].to_numpy()
_n = 6000
_peso = np.array([0.2, 0.17, 0.15, 0.1, 0.14, 0.09, 0.15])
_ventas = pd.DataFrame({
    "id_venta": range(1, _n + 1),
    "fecha": (pd.Timestamp("2025-01-01") + pd.to_timedelta(rng.integers(0, 181, _n), unit="D")).strftime("%Y-%m-%d"),
    "id_tienda": rng.choice(np.arange(1, 8), _n, p=_peso / _peso.sum()),
    "id_producto": rng.choice(_vendibles, _n),
    "unidades": 1 + rng.poisson(1.5, _n),
    "descuento": rng.choice([0.0, 0.05, 0.1], _n, p=[0.7, 0.2, 0.1]),
})
_ventas = _ventas.sort_values(["fecha", "id_venta"]).reset_index(drop=True)
_ventas["id_venta"] = range(1, _n + 1)

# Base de prueba pequeña, con casos borde: una tienda sin ventas, un producto sin ventas, un empate y un mes sin ventas.
_tiendas_p = pd.DataFrame({"id_tienda": [1, 2, 3], "nombre": ["Tienda A", "Tienda B", "Tienda C"],
                           "region": ["Lima", "Lima", "Norte"], "apertura": ["2020-01-01", "2021-01-01", "2025-06-01"]})
_productos_p = pd.DataFrame({"id_producto": [1, 2, 3, 4], "nombre": ["Producto X", "Producto Y", "Producto Z", "Producto W"],
                             "categoria": ["Uno", "Uno", "Dos", "Dos"], "precio": [10.0, 5.0, 2.0, 7.0]})
_ventas_p = pd.DataFrame({"id_venta": [1, 2, 3, 4, 5], "fecha": ["2025-01-05", "2025-01-20", "2025-03-02", "2025-03-15", "2025-03-31"],
                          "id_tienda": [1, 2, 1, 2, 1], "id_producto": [1, 2, 3, 3, 3], "unidades": [1, 2, 5, 6, 1],
                          "descuento": [0.0, 0.0, 0.1, 0.0, 0.05]})


def _crear_base(t, p, v):
    c = sqlite3.connect(":memory:")
    t.to_sql("tiendas", c, index=False)
    p.to_sql("productos", c, index=False)
    v.to_sql("ventas", c, index=False)
    return c


con = _crear_base(_tiendas, _productos, _ventas)
_CON_PRUEBA = _crear_base(_tiendas_p, _productos_p, _ventas_p)
_BASES = [("la base del curso", lambda: con, (_tiendas, _productos, _ventas)),
          ("una base de prueba pequeña (con una tienda sin ventas, un producto sin ventas, un empate y un mes sin ventas)",
           lambda: _CON_PRUEBA, (_tiendas_p, _productos_p, _ventas_p))]

# ---------- Herramientas de verificación ----------
_FALTA = object()


def _h(valor):
    if isinstance(valor, str):
        valor = valor.strip().lower()
    return hashlib.sha256(f"{type(valor).__name__}|{valor!r}".encode("utf-8")).hexdigest()


def _cerca(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=1e-9, abs_tol=tol)


def _dos_decimales(x):
    return abs(x * 100 - round(x * 100)) < 1e-6


def _corto(valor, n=60):
    if type(valor).__module__ == "numpy" and getattr(valor, "shape", None) == ():
        valor = valor.item()
    texto = repr(valor)
    return texto if len(texto) <= n else texto[:n] + "…"


def _igual(a, b, tol=1e-6):
    """Compara exigiendo el mismo tipo en None/bool y tolerancia en decimales."""
    if b is None or isinstance(b, bool):
        return type(a) is type(b) and a == b
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return (isinstance(a, (int, float)) and not isinstance(a, bool)
                and math.isclose(a, b, rel_tol=1e-9, abs_tol=tol))
    if isinstance(b, (list, tuple)):
        return (type(a) is type(b) and len(a) == len(b)
                and all(_igual(x, y, tol) for x, y in zip(a, b)))
    if isinstance(b, dict):
        return (isinstance(a, dict) and set(a) == set(b)
                and all(_igual(a[k], b[k], tol) for k in b))
    return type(a) is type(b) and a == b


class _Revision:
    def __init__(self, titulo):
        self.titulo = titulo
        self.errores = 0
        print(f"── {titulo} ──")

    def ok(self, msg):
        print(f"✅ {msg}")

    def mal(self, msg):
        self.errores += 1
        print(f"❌ {msg}")

    def var(self, nombre, tipo=None):
        valor = globals().get(nombre, _FALTA)
        if valor is _FALTA:
            self.mal(f"No encuentro `{nombre}`. ¿Ejecutaste tu celda? ¿Escribiste bien el nombre?")
            return _FALTA
        if tipo is not None and not (type(valor) is tipo or (isinstance(tipo, tuple) and type(valor) in tipo)):
            esperado = tipo.__name__ if not isinstance(tipo, tuple) else " o ".join(t.__name__ for t in tipo)
            self.mal(f"`{nombre}` es de tipo {type(valor).__name__} y se esperaba {esperado}.")
            return _FALTA
        return valor

    def funcion(self, nombre):
        f = self.var(nombre)
        if f is _FALTA:
            return _FALTA
        if not callable(f):
            self.mal(f"`{nombre}` existe pero no es una función. ¿La definiste con `def`?")
            return _FALTA
        return f

    def caso(self, texto, f, args=(), kwargs=None, esperado=None, igual=None, motivo="no es lo esperado", tol=0.0051):
        """Llama a f con copias de los argumentos y compara sin mostrar el valor esperado."""
        import copy
        try:
            obtenido = f(*copy.deepcopy(args), **copy.deepcopy(kwargs or {}))
        except Exception as e:
            self.mal(f"`{texto}` lanzó {type(e).__name__}: {e}")
            return False
        bien = igual(obtenido, esperado) if igual else _igual(obtenido, esperado, tol)
        if bien:
            self.ok(f"`{texto}` funciona.")
        else:
            self.mal(f"`{texto}` devolvió {_corto(obtenido)}; {motivo}.")
        return bien

    def valor(self, nombre, esperado, tipo=None, pista="revisa el cálculo", igual=None):
        v = self.var(nombre, tipo)
        if v is _FALTA:
            return
        bien = igual(v, esperado) if igual else _igual(v, esperado)
        if bien:
            self.ok(f"`{nombre}` es correcto.")
        else:
            self.mal(f"`{nombre}` vale {_corto(v)}; {pista}.")

    def texto_limpio(self, nombre, valor):
        if valor != valor.strip():
            self.mal(f"`{nombre}` tiene espacios o saltos de línea al inicio o al final: {valor!r}")
            return False
        return True

    def predicciones(self, esperados):
        for nombre, hash_ok in esperados.items():
            v = self.var(nombre)
            if v is _FALTA:
                continue
            if _h(v) == hash_ok:
                self.ok(f"`{nombre}` es correcto.")
            else:
                self.mal(f"`{nombre}` no es correcto. Razónalo otra vez y luego compruébalo ejecutando la expresión en una celda nueva.")

    def fin(self):
        if self.errores == 0:
            print(f"🎉 ¡{self.titulo} superado!")
        else:
            cuantos = "el punto marcado" if self.errores == 1 else f"los {self.errores} puntos marcados"
            print(f"🔁 Corrige {cuantos} con ❌ y vuelve a verificar.")


def _primera_diferencia(r, nombre, tuyo, esperado):
    for i, (a, b) in enumerate(zip(tuyo, esperado)):
        if a != b:
            r.mal(f"`{nombre}` no tiene el formato pedido. La diferencia empieza en el carácter {i}: "
                  f"desde ahí tu texto dice {tuyo[i:i + 15]!r}.")
            return
    n = abs(len(esperado) - len(tuyo))
    cuantos = "1 carácter" if n == 1 else f"{n} caracteres"
    if len(tuyo) < len(esperado):
        r.mal(f"`{nombre}` está incompleto: le {'falta' if n == 1 else 'faltan'} {cuantos} al final.")
    else:
        r.mal(f"`{nombre}` tiene {cuantos} de más al final: {tuyo[len(esperado):]!r}.")


def _es_numero(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_))


def _esc(r, nombre, esperado, pista, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if isinstance(v, np.ndarray) and v.shape == ():
        v = v.item()
    if not _es_numero(v):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un número.")
    elif abs(float(v) - esperado) <= tol:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` vale {_corto(v.item() if hasattr(v, 'item') else v)}; {pista}.")


def _arr(r, nombre, esperado, pista, tol=1e-6, tipos=None):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, np.ndarray):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un array de NumPy (`np.ndarray`).")
        return
    esperado = np.array(esperado)
    if v.shape != esperado.shape:
        r.mal(f"`{nombre}` tiene forma {v.shape} y se esperaba {esperado.shape}.")
        return
    if tipos and v.dtype.kind not in tipos:
        nombres = {"b": "bool", "i": "entero", "u": "entero", "f": "decimal (float)", "U": "texto"}
        r.mal(f"`{nombre}` tiene dtype {v.dtype} y se esperaba un tipo {' o '.join(sorted({nombres[t] for t in tipos}))}.")
        return
    if v.dtype.kind in "USO" or esperado.dtype.kind in "USO":
        bien = v.tolist() == esperado.tolist()
    elif v.dtype.kind == "b" or esperado.dtype.kind == "b":
        bien = np.array_equal(v, esperado)
    else:
        bien = np.allclose(v.astype(float), esperado.astype(float), rtol=0, atol=tol, equal_nan=True)
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")




def _sin_cambios(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        original = _D[n]
        if isinstance(original, np.ndarray):
            igual = isinstance(actual, np.ndarray) and actual.shape == original.shape and np.array_equal(actual, original, equal_nan=original.dtype.kind == "f")
        else:
            igual = actual == original
        if not igual:
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a ejecutar el setup.")


def _norm(x):
    if isinstance(x, np.generic):
        x = x.item()
    try:
        if pd.isna(x):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(x, pd.Timestamp):
        return str(x)
    return x


def _mismo(a, b, tol=1e-6):
    a, b = _norm(a), _norm(b)
    if a is None or b is None:
        return a is None and b is None
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return isinstance(a, (int, float)) and not isinstance(a, bool) and abs(a - b) <= tol
    return str(a) == str(b)


def _ser(r, nombre, valores, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.Series):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba una Series de pandas.")
        return
    if len(v) != len(valores):
        r.mal(f"`{nombre}` tiene {len(v)} elementos y se esperaban {len(valores)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    if all(_mismo(a, b, tol) for a, b in zip(v.tolist(), valores)):
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene el largo correcto pero sus valores no coinciden; {pista}.")


def _df(r, nombre, columnas, filas, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.DataFrame):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un DataFrame de pandas.")
        return
    cols = [str(c) for c in v.columns]
    if cols != columnas:
        faltan = [c for c in columnas if c not in cols]
        sobran = [c for c in cols if c not in columnas]
        if faltan or sobran:
            r.mal(f"A `{nombre}` le faltan las columnas {faltan} y le sobran {sobran}." if faltan and sobran else
                  (f"A `{nombre}` le faltan las columnas {faltan}." if faltan else f"En `{nombre}` sobran las columnas {sobran}."))
        else:
            r.mal(f"`{nombre}` tiene las columnas correctas pero en otro orden.")
        return
    if len(v) != len(filas):
        r.mal(f"`{nombre}` tiene {len(v)} filas y se esperaban {len(filas)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas de fila (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    bien = all(_mismo(a, b, tol) for fila_v, fila_e in zip(v.itertuples(index=False), filas) for a, b in zip(fila_v, fila_e))
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")


def _sin_cambios_df(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        if not (isinstance(actual, (pd.DataFrame, pd.Series)) and actual.equals(_D[n])):
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a cargarlos o ejecuta de nuevo el setup.")

def _hex(c):
    return mcolors.to_hex(c).lower()


def _texto(a, b):
    return isinstance(a, str) and a.strip().lower() == b.strip().lower()


def _grafico(r, nombre):
    ax = r.var(nombre)
    if ax is _FALTA:
        return None
    if not isinstance(ax, mpl.axes.Axes):
        r.mal(f"`{nombre}` es de tipo {type(ax).__name__} y se esperaba un eje de Matplotlib (lo que devuelve `plt.subplots()`).")
        return None
    return ax


def _rotulos(r, nombre, ax, titulo=None, xlabel=None, ylabel=None):
    titulo_actual = next((t for t in (ax.get_title(loc=l) for l in ("left", "center", "right")) if t.strip()), "")
    for que, genero, obtenido, esperado in (("el título", "correcto", titulo_actual, titulo),
                                            ("la etiqueta del eje x", "correcta", ax.get_xlabel(), xlabel),
                                            ("la etiqueta del eje y", "correcta", ax.get_ylabel(), ylabel)):
        if esperado is None:
            continue
        if _texto(obtenido, esperado):
            r.ok(f"En `{nombre}`, {que} es {genero}.")
        elif not obtenido.strip():
            r.mal(f"A `{nombre}` le falta {que}.")
        else:
            r.mal(f"En `{nombre}`, {que} dice {obtenido!r}; revisa el texto pedido.")


def _barras(ax):
    """Rectángulos de barras (sin el fondo del eje), en orden de dibujo."""
    return [p for p in ax.patches if isinstance(p, mpl.patches.Rectangle)]


def _cerca_lista(a, b, tol=1e-6):
    return len(a) == len(b) and all(abs(float(x) - float(y)) <= tol for x, y in zip(a, b))


def _formato(ax, eje, valor):
    fmt = (ax.yaxis if eje == "y" else ax.xaxis).get_major_formatter()
    return fmt(valor, 0)


def _detalle(t, p, v):
    d = v.merge(p, on="id_producto").merge(t, on="id_tienda", suffixes=("_producto", "_tienda"))
    d["ingreso"] = d["unidades"] * d["precio"] * (1 - d["descuento"])
    return d.sort_values("id_venta")


def _ref_1(t, p, v):
    d = v[(v["fecha"] >= "2025-03-01") & (v["fecha"] <= "2025-03-31") & (v["unidades"] >= 5)]
    d = sorted(d.itertuples(index=False), key=lambda f: (-f.unidades, f.id_venta))
    return [[f.id_venta, f.fecha, f.id_tienda, f.unidades] for f in d]


def _ref_2(t, p, v):
    d = _detalle(t, p, v)
    return [[f.id_venta, f.fecha, f.nombre_tienda, f.region, f.nombre_producto, f.categoria, f.ingreso] for f in d.itertuples(index=False)]


def _ref_sin_ventas(t, p, v):
    vendidos = set(v["id_producto"])
    return sorted([[f.nombre, f.categoria] for f in p.itertuples(index=False) if f.id_producto not in vendidos])


def _por_tienda(t, p, v):
    d = _detalle(t, p, v)
    res = {}
    for f in d.itertuples(index=False):
        x = res.setdefault(f.id_tienda, [f.nombre_tienda, f.region, 0, 0.0])
        x[2] += 1
        x[3] += f.ingreso
    return list(res.values())


def _ref_3(t, p, v):
    return sorted([x for x in _por_tienda(t, p, v) if x[3] > 15000], key=lambda x: -x[3])


def _ref_4(t, p, v):
    filas = _por_tienda(t, p, v)
    prom = {}
    for x in filas:
        prom.setdefault(x[1], []).append(x[3])
    res = [[x[0], x[1], x[3], statistics.fmean(prom[x[1]])] for x in filas if x[3] > statistics.fmean(prom[x[1]])]
    return sorted(res, key=lambda x: (x[1], -x[2]))


def _ref_5(t, p, v):
    d = _detalle(t, p, v)
    meses = {}
    for f in d.itertuples(index=False):
        meses[f.fecha[:7]] = meses.get(f.fecha[:7], 0.0) + f.ingreso
    res, acum, previo = [], 0.0, None
    for m in sorted(meses):
        acum += meses[m]
        res.append([m, meses[m], acum, None if previo is None else meses[m] - previo])
        previo = meses[m]
    return res


def _ref_reto(t, p, v):
    d = _detalle(t, p, v)
    tot = {}
    for f in d.itertuples(index=False):
        tot[(f.categoria, f.nombre_producto)] = tot.get((f.categoria, f.nombre_producto), 0.0) + f.ingreso
    res = []
    for cat in sorted({c for c, _ in tot}):
        prods = [(n, x) for (c, n), x in tot.items() if c == cat]
        for n, x in prods:
            puesto = 1 + sum(1 for _, y in prods if y > x + 1e-9)
            if puesto <= 2:
                res.append([cat, n, x, puesto])
    return sorted(res, key=lambda r: (r[0], r[3], r[1]))


def _base_intacta(r):
    try:
        n, s = con.execute("SELECT COUNT(*), SUM(unidades) FROM ventas").fetchone()
        ok = (n, s) == (len(_ventas), int(_ventas["unidades"].sum())) and con.execute("SELECT COUNT(*) FROM tiendas").fetchone()[0] == 8
    except Exception:
        ok = False
    if not ok:
        r.mal("La base `con` cambió. No modifiques sus tablas; vuelve a ejecutar el setup.")
    return ok


def _comparar(r, nombre, df, filas, columnas, donde):
    if [str(c) for c in df.columns] != columnas:
        r.mal(f"`{nombre}` en {donde} devuelve las columnas {[str(c) for c in df.columns]} y se esperaban {columnas}. Usa `AS` para nombrarlas.")
        return False
    if len(df) != len(filas):
        r.mal(f"`{nombre}` en {donde} devuelve {len(df)} filas y se esperaban {len(filas)}.")
        return False
    for i, (fila_v, fila_e) in enumerate(zip(df.itertuples(index=False), filas)):
        if not all(_mismo(a, b, 1e-4) for a, b in zip(fila_v, fila_e)):
            r.mal(f"`{nombre}` en {donde}: la fila {i} no coincide con lo esperado; revisa los filtros, los cálculos y el `ORDER BY`.")
            return False
    return True


def _revisar_consulta(r, nombre, resultado, ref, columnas):
    q = r.var(nombre, str)
    if q is _FALTA:
        return
    bien = True
    for donde, conexion, datos in _BASES:
        try:
            df = pd.read_sql(q, conexion())
        except Exception as ex:
            r.mal(f"`{nombre}` falla en {donde}: {type(ex).__name__}: {str(ex)[:120]}")
            return
        if not _comparar(r, nombre, df, ref(*datos), columnas, donde):
            bien = False
            break
    if bien:
        r.ok(f"`{nombre}` da el resultado correcto, también en la base de prueba.")
        v = r.var(resultado)
        if v is not _FALTA:
            if not isinstance(v, pd.DataFrame) or not _comparar(r, resultado, v, ref(*_BASES[0][2]), columnas, "la base del curso"):
                if not isinstance(v, pd.DataFrame):
                    r.mal(f"`{resultado}` debería ser el DataFrame de `pd.read_sql({nombre}, con)`.")
            else:
                r.ok(f"`{resultado}` es correcto.")


def check_ejercicio_1():
    r = _Revision("Ejercicio 1 · Parte A")
    if _base_intacta(r):
        _revisar_consulta(r, "consulta_1", "ventas_marzo", _ref_1, ["id_venta", "fecha", "id_tienda", "unidades"])
        f = r.funcion("ventas_de_tienda")
        if f is not _FALTA:
            for id_t in (3, 8, 99):
                try:
                    res = f(id_t)
                except Exception as ex:
                    r.mal(f"`ventas_de_tienda({id_t})` lanzó {type(ex).__name__}: {ex}")
                    continue
                esperado = [[x.id_venta, x.fecha, x.unidades] for x in _ventas[_ventas["id_tienda"] == id_t].sort_values("id_venta").itertuples(index=False)]
                if not isinstance(res, pd.DataFrame):
                    r.mal(f"`ventas_de_tienda({id_t})` debería devolver un DataFrame.")
                elif _comparar(r, f"ventas_de_tienda({id_t})", res, esperado, ["id_venta", "fecha", "unidades"], "la base del curso"):
                    r.ok(f"`ventas_de_tienda({id_t})` funciona" + (" (sin ventas: 0 filas)." if not esperado else "."))
    r.fin()
    r = _Revision("Ejercicio 1 · Parte B")
    r.predicciones({
        "pred_filas_minuscula": "25af31a57c2047d854d189042b0ecfb66843c4c19d3dd9ce1403e0be3826a240",
    })
    r.fin()


def check_ejercicio_2():
    r = _Revision("Ejercicio 2 · Parte A")
    if _base_intacta(r):
        _revisar_consulta(r, "consulta_2", "detalle", _ref_2, ["id_venta", "fecha", "tienda", "region", "producto", "categoria", "ingreso"])
        _revisar_consulta(r, "consulta_sin_ventas", "sin_ventas", _ref_sin_ventas, ["producto", "categoria"])
    r.fin()
    r = _Revision("Ejercicio 2 · Parte B")
    r.predicciones({
        "pred_inner_incluye": "e675bdd897ba87a607b7c344f97a5152cda452c1b1641515ea9436fd35397ada",
    })
    r.fin()


def check_ejercicio_3():
    r = _Revision("Ejercicio 3 · Parte A")
    if _base_intacta(r):
        _revisar_consulta(r, "consulta_3", "ranking_tiendas", _ref_3, ["tienda", "region", "n_ventas", "ingreso"])
    r.fin()
    r = _Revision("Ejercicio 3 · Parte B")
    r.predicciones({
        "pred_where_suma": "e675bdd897ba87a607b7c344f97a5152cda452c1b1641515ea9436fd35397ada",
    })
    r.fin()


def check_ejercicio_4():
    r = _Revision("Ejercicio 4 · Parte A")
    if _base_intacta(r):
        q = globals().get("consulta_4")
        if isinstance(q, str) and not q.strip().upper().startswith("WITH"):
            r.mal("`consulta_4` debería empezar con `WITH`: arma los pasos intermedios como CTE.")
        else:
            _revisar_consulta(r, "consulta_4", "sobre_promedio", _ref_4, ["tienda", "region", "ingreso", "promedio_region"])
    r.fin()
    r = _Revision("Ejercicio 4 · Parte B")
    r.predicciones({
        "pred_cte_guarda": "e675bdd897ba87a607b7c344f97a5152cda452c1b1641515ea9436fd35397ada",
    })
    r.fin()


def check_ejercicio_5():
    r = _Revision("Ejercicio 5 · Parte A")
    if _base_intacta(r):
        _revisar_consulta(r, "consulta_5", "mensual", _ref_5, ["mes", "ingreso", "acumulado", "variacion"])
        ax = _grafico(r, "ax_mensual")
        if ax is not None:
            ref = _ref_5(_tiendas, _productos, _ventas)
            barras = _barras(ax)
            if len(barras) != len(ref) or not _cerca_lista([b.get_height() for b in barras], [x[1] for x in ref], 1e-4):
                r.mal("`ax_mensual` debería tener una barra vertical por mes con su ingreso.")
            else:
                r.ok("`ax_mensual` muestra el ingreso de cada mes.")
            _rotulos(r, "ax_mensual", ax, None, None, "Ingreso (S/)")
    r.fin()
    r = _Revision("Ejercicio 5 · Parte B")
    r.predicciones({
        "pred_lag_primero": "31d7c71a1e9c0b87b8f6ac118378c7caf62457a084c838db5b6814568e91e5dd",
    })
    r.fin()


def check_reto():
    r = _Revision("Reto final")
    if _base_intacta(r):
        _revisar_consulta(r, "consulta_reto", "top_productos", _ref_reto, ["categoria", "producto", "ingreso", "puesto"])
    r.fin()


def check_pro():
    r = _Revision("Nivel pro")
    ref = _ref_5(_tiendas, _productos, _ventas)
    try:
        tabla = pd.read_sql("SELECT * FROM resumen_mensual", con)
    except Exception:
        tabla = None
    if tabla is None:
        r.mal("No encuentro la tabla `resumen_mensual` en `con`. Créala con `mensual.to_sql(\"resumen_mensual\", con, index=False, if_exists=\"replace\")`.")
    elif _comparar(r, "resumen_mensual", tabla, ref, ["mes", "ingreso", "acumulado", "variacion"], "la base del curso"):
        r.ok("La tabla `resumen_mensual` está guardada en la base.")
        q = r.var("consulta_pro", str)
        if q is not _FALTA:
            prom = statistics.fmean([x[1] for x in ref])
            esperado = [[x[0], x[1]] for x in ref if x[1] > prom]
            try:
                df = pd.read_sql(q, con)
            except Exception as ex:
                r.mal(f"`consulta_pro` falla: {type(ex).__name__}: {str(ex)[:120]}")
            else:
                if _comparar(r, "consulta_pro", df, esperado, ["mes", "ingreso"], "la base del curso"):
                    r.ok("`consulta_pro` es correcta.")
    r.fin()


print("✅ Setup listo. Base de datos creada (`con`), estilo aplicado y verificadores cargados.")

### 📦 Tu base de datos de hoy
`con` es una conexión a una base SQLite con tres tablas de una cadena de tiendas:
- **`tiendas`**: `id_tienda`, `nombre`, `region`, `apertura`.
- **`productos`**: `id_producto`, `nombre`, `categoria`, `precio` (en soles).
- **`ventas`**: `id_venta`, `fecha` (texto `AAAA-MM-DD`), `id_tienda`, `id_producto`, `unidades`, `descuento` (0, 0.05 o 0.1).

El **ingreso** de una venta es `unidades * precio * (1 - descuento)`. En SQLite las fechas se guardan como texto; como el formato es `AAAA-MM-DD`, se comparan y ordenan bien como texto.

In [ ]:
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", con))
print(pd.read_sql("SELECT * FROM tiendas", con))
print(pd.read_sql("SELECT * FROM ventas LIMIT 5", con))

---
## 1. Consultar desde Python: `SELECT`, `WHERE`, `ORDER BY`

### 📘 Concepto
`sqlite3` viene con Python. `sqlite3.connect("archivo.db")` abre (o crea) una base; `":memory:"` la crea en memoria. Con `pd.read_sql(consulta, con)` el resultado llega como DataFrame.

```sql
SELECT columna1, columna2 AS otro_nombre   -- qué columnas
FROM tabla                                 -- de dónde
WHERE condicion AND otra_condicion         -- qué filas
ORDER BY columna1 DESC, columna2           -- en qué orden
LIMIT 10                                   -- cuántas
```

Los textos van entre comillas simples, y las comparaciones de texto distinguen mayúsculas. Cuando un valor viene de una variable, **no lo pegues** dentro del texto de la consulta: usa un parámetro `?` y pásalo con `params`. Así evitas errores y la inyección de SQL.

In [ ]:
consulta_ej = """
SELECT nombre, precio
FROM productos
WHERE categoria = ? AND precio < 10
ORDER BY precio DESC
"""
print(pd.read_sql(consulta_ej, con, params=("Lácteos",)))

### ✍️ Tu turno · Ejercicio 1: tus primeras consultas
**Parte A.**
1. `consulta_1`: las ventas de **marzo de 2025** con 5 unidades o más, con las columnas `id_venta`, `fecha`, `id_tienda` y `unidades`, ordenadas de más a menos unidades y, a igual cantidad, por `id_venta`. `ventas_marzo = pd.read_sql(consulta_1, con)`.
2. `ventas_de_tienda(id_tienda)`: una función que devuelve un DataFrame con `id_venta`, `fecha` y `unidades` de las ventas de esa tienda, ordenadas por `id_venta`, usando un parámetro `?`. Pruébala con una tienda que no tiene ventas.

**Parte B.** Predice **sin ejecutar**: ¿cuántas filas devuelve `SELECT * FROM tiendas WHERE region = 'lima'`? Guárdalo en `pred_filas_minuscula` (un entero).

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_1()

<details><summary>💡 Pista 1</summary>

Para marzo: `fecha BETWEEN '2025-03-01' AND '2025-03-31'`. Recuerda que el orden va con `ORDER BY unidades DESC, id_venta`.
</details>

<details><summary>💡 Pista 2</summary>

Dentro de la función: `return pd.read_sql("SELECT ... WHERE id_tienda = ? ORDER BY id_venta", con, params=(id_tienda,))`. Ojo con la coma: `(id_tienda,)` es una tupla.
</details>

---
## 2. Combinar tablas: `JOIN` y `LEFT JOIN`

### 📘 Concepto
`JOIN` une filas de dos tablas que coinciden en una columna, como `merge` en pandas:

```sql
SELECT v.id_venta, p.nombre AS producto
FROM ventas AS v
JOIN productos AS p ON p.id_producto = v.id_producto
```

- `JOIN` (o `INNER JOIN`) deja solo las filas que tienen pareja en ambas tablas.
- `LEFT JOIN` deja **todas** las filas de la tabla de la izquierda; si no tienen pareja, las columnas de la derecha quedan en `NULL`. Con `WHERE derecha.columna IS NULL` encuentras lo que **no** tiene pareja (por ejemplo, productos que nunca se vendieron).

Los alias (`v`, `p`) acortan la consulta y evitan ambigüedades cuando dos tablas tienen una columna con el mismo nombre (aquí, `nombre`). Los cálculos se escriben directamente: `v.unidades * p.precio AS bruto`.

In [ ]:
print(pd.read_sql("""
SELECT t.nombre AS tienda, v.id_venta
FROM tiendas AS t
LEFT JOIN ventas AS v ON v.id_tienda = t.id_tienda
WHERE v.id_venta IS NULL
""", con))

### ✍️ Tu turno · Ejercicio 2: unir ventas, productos y tiendas
**Parte A.**
1. `consulta_2`: una fila por venta con `id_venta`, `fecha`, `tienda` (nombre de la tienda), `region`, `producto` (nombre del producto), `categoria` e `ingreso`, ordenada por `id_venta`. `detalle = pd.read_sql(consulta_2, con)`.
2. `consulta_sin_ventas`: los productos que nunca se vendieron, con las columnas `producto` y `categoria`, ordenados por `producto`. `sin_ventas = pd.read_sql(consulta_sin_ventas, con)`.

**Parte B.** Responde en `pred_inner_incluye` con `"sí"` o `"no"`: un `JOIN` (inner) entre `productos` y `ventas`, ¿incluye los productos sin ventas?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_2()

<details><summary>💡 Pista 1</summary>

Necesitas dos `JOIN` seguidos: `ventas` con `productos` y `ventas` con `tiendas`. Nombra las columnas con `AS`.
</details>

<details><summary>💡 Pista 2</summary>

Para los productos sin ventas: `FROM productos AS p LEFT JOIN ventas AS v ON ... WHERE v.id_venta IS NULL`.
</details>

---
## 3. Resumir: `GROUP BY` y `HAVING`

### 📘 Concepto
`GROUP BY` agrupa filas y las funciones de agregación resumen cada grupo: `COUNT(*)`, `SUM(...)`, `AVG(...)`, `MIN(...)`, `MAX(...)`.

- `WHERE` filtra **filas antes** de agrupar;
- `HAVING` filtra **grupos después** de agrupar, y puede usar agregaciones.

El orden de las cláusulas es fijo: `SELECT … FROM … JOIN … WHERE … GROUP BY … HAVING … ORDER BY … LIMIT`.

In [ ]:
print(pd.read_sql("""
SELECT p.categoria, COUNT(*) AS n_ventas, SUM(v.unidades) AS unidades
FROM ventas AS v
JOIN productos AS p ON p.id_producto = v.id_producto
WHERE v.descuento > 0
GROUP BY p.categoria
HAVING COUNT(*) > 500
ORDER BY unidades DESC
""", con))

### ✍️ Tu turno · Ejercicio 3: las tiendas que más venden
**Parte A.** `consulta_3`: una fila por tienda con `tienda`, `region`, `n_ventas` (cantidad de ventas) e `ingreso` (suma del ingreso), **solo** las tiendas con un ingreso mayor a 15 000 soles, de mayor a menor ingreso. `ranking_tiendas = pd.read_sql(consulta_3, con)`.

**Parte B.** Responde en `pred_where_suma` con `"sí"` o `"no"`: ¿puedes filtrar por `SUM(...)` dentro de `WHERE`?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_3()

<details><summary>💡 Pista 1</summary>

Agrupa por `t.id_tienda` (y, para mostrarlas, por `t.nombre` y `t.region`).
</details>

<details><summary>💡 Pista 2</summary>

`HAVING SUM(v.unidades * p.precio * (1 - v.descuento)) > 15000`. En SQLite también puedes usar el alias: `HAVING ingreso > 15000`.
</details>

---
## 4. Consultas en pasos: CTE con `WITH`

### 📘 Concepto
Cuando una consulta tiene varios pasos, una **CTE** (*common table expression*) le pone nombre a cada paso, como variables intermedias:

```sql
WITH por_tienda AS (
    SELECT ... GROUP BY ...
),
por_region AS (
    SELECT region, AVG(ingreso) AS promedio FROM por_tienda GROUP BY region
)
SELECT ...
FROM por_tienda
JOIN por_region ON ...
```

Cada paso puede usar los anteriores. La CTE existe solo mientras corre la consulta: no crea tablas en la base.

In [ ]:
print(pd.read_sql("""
WITH caros AS (
    SELECT * FROM productos WHERE precio > 8
)
SELECT categoria, COUNT(*) AS productos_caros
FROM caros
GROUP BY categoria
""", con))

### ✍️ Tu turno · Ejercicio 4: tiendas sobre el promedio de su región
**Parte A.** `consulta_4`: una consulta que empiece con `WITH` y devuelva las tiendas cuyo ingreso supera el **promedio de ingreso de las tiendas de su región** (solo tiendas con ventas), con las columnas `tienda`, `region`, `ingreso` y `promedio_region`, ordenadas por `region` y luego de mayor a menor ingreso. `sobre_promedio = pd.read_sql(consulta_4, con)`.

**Parte B.** Responde en `pred_cte_guarda` con `"sí"` o `"no"`: después de correr tu consulta, ¿queda guardada en la base una tabla con el nombre de tu CTE?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_4()

<details><summary>💡 Pista 1</summary>

El primer paso es casi tu `consulta_3` sin el `HAVING`. El segundo calcula el promedio por región a partir del primero.
</details>

<details><summary>💡 Pista 2</summary>

Termina con `SELECT ... FROM por_tienda AS a JOIN por_region AS b ON a.region = b.region WHERE a.ingreso > b.promedio_region ORDER BY ...`.
</details>

---
## 5. Funciones de ventana

### 📘 Concepto
Una **función de ventana** calcula algo sobre un grupo de filas **sin colapsarlas**, a diferencia de `GROUP BY`. Se escribe con `OVER (...)`:
- `SUM(x) OVER (ORDER BY mes)`: suma acumulada hasta cada fila;
- `LAG(x) OVER (ORDER BY mes)`: el valor de la fila anterior (en la primera no hay anterior: da `NULL`);
- `RANK() OVER (PARTITION BY grupo ORDER BY x DESC)`: el puesto dentro de cada grupo (empates comparten puesto).

`PARTITION BY` reinicia el cálculo en cada grupo. Una función de ventana no puede ir en `WHERE`: si quieres filtrar por ella, calcúlala en una CTE y filtra después. Para el mes, `strftime('%Y-%m', fecha)` extrae año y mes de una fecha en texto.

In [ ]:
print(pd.read_sql("""
SELECT nombre, categoria, precio,
       RANK() OVER (PARTITION BY categoria ORDER BY precio DESC) AS puesto,
       AVG(precio) OVER (PARTITION BY categoria) AS precio_medio_categoria
FROM productos
ORDER BY categoria, puesto
""", con).head(8))

### ✍️ Tu turno · Ejercicio 5: ingreso mensual con acumulado y variación
**Parte A.**
1. `consulta_5`: una fila por mes con `mes` (texto `AAAA-MM`), `ingreso` (del mes), `acumulado` (suma acumulada del ingreso) y `variacion` (ingreso del mes menos el del mes anterior), ordenada por `mes`. `mensual = pd.read_sql(consulta_5, con)`.
2. `fig_mensual, ax_mensual`: barras verticales con el ingreso de cada mes, eje y `Ingreso (S/)` y un título-conclusión.

**Parte B.** Responde en `pred_lag_primero` con `"cero"` o `"nulo"`: ¿qué da `variacion` en el primer mes?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_5()

<details><summary>💡 Pista 1</summary>

Primero agrupa por mes en una CTE; después, en el `SELECT` final, aplica las funciones de ventana sobre esa CTE.
</details>

<details><summary>💡 Pista 2</summary>

`SUM(ingreso) OVER (ORDER BY mes) AS acumulado` y `ingreso - LAG(ingreso) OVER (ORDER BY mes) AS variacion`.
</details>

---
## 🏋️ Reto final: los dos productos estrella de cada categoría
`consulta_reto`: para cada categoría, los productos en los **puestos 1 y 2** por ingreso total (con `RANK`, así que un empate puede dejar más de dos), con las columnas `categoria`, `producto`, `ingreso` y `puesto`, ordenados por `categoria`, `puesto` y `producto`. `top_productos = pd.read_sql(consulta_reto, con)`.

¿Qué categoría depende más de un solo producto?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_reto()

<details><summary>💡 Pista 1</summary>

Paso 1 (CTE): ingreso por producto y categoría. Paso 2 (CTE): agrega el `RANK() OVER (PARTITION BY categoria ORDER BY ingreso DESC)`. Paso 3: filtra `puesto <= 2`.
</details>

<details><summary>💡 Pista 2</summary>

No puedes usar `puesto` en el `WHERE` del mismo `SELECT` que lo calcula: por eso va en otra CTE.
</details>

---
## 🚀 Nivel pro (opcional): de pandas a SQL
También puedes escribir un DataFrame en la base con `df.to_sql("nombre", con, index=False, if_exists="replace")`. Guarda `mensual` como la tabla `resumen_mensual` y escribe `consulta_pro`: los meses (columnas `mes` e `ingreso`) cuyo ingreso supera el promedio mensual, calculado con una subconsulta `(SELECT AVG(ingreso) FROM resumen_mensual)`, ordenados por `mes`.

Este es el paso que harás en tu proyecto: tablas limpias de pandas guardadas en una base SQL.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_pro()

---
## 🧱 Avance del proyecto: P4 · tu base SQL

**Qué hacer**
1. En `notebooks/05_sql.ipynb`, crea una base SQLite (`data/proyecto.db`) con las tablas limpias de P2: una tabla de hechos (una fila por reclamo o por registro) y tablas de dimensión (entidades, regiones, productos o motivos), con claves que las unan. Agrega también la tabla de segmentos de S24.
2. No subas el archivo `.db` si pesa mucho: el notebook debe poder recrearlo desde los datos descargados.
3. Responde con SQL al menos cuatro preguntas del proyecto, usando `JOIN`, `GROUP BY` con `HAVING`, una CTE y una función de ventana (por ejemplo, el ranking de entidades por reclamos cada 10 000 clientes dentro de cada año, o la variación anual).
4. Compara uno de esos resultados con el que obtuviste en pandas: deben coincidir.
5. Guarda las consultas finales en un archivo `sql/consultas.sql` con un comentario por consulta que diga qué pregunta responde.

**Por qué lo haría un analista**
En casi cualquier empresa los datos viven en una base SQL. Mostrar que sabes modelar tablas y responder preguntas de negocio con consultas claras es de lo primero que se evalúa en una entrevista de analista.

**Cómo debe verse el resultado**
Un notebook que crea la base desde cero, un archivo de consultas legible y comentado, y resultados que coinciden con tu análisis en pandas. Estas tablas alimentarán tu dashboard (S27).

---
## ✅ Cierre: autoevaluación
Marca lo que puedes hacer sin mirar:
- [ ] Conectarme a SQLite y traer una consulta a pandas con `pd.read_sql`.
- [ ] Usar parámetros `?` en lugar de pegar valores en la consulta.
- [ ] Explicar la diferencia entre `JOIN` y `LEFT JOIN`, y encontrar filas sin pareja.
- [ ] Explicar la diferencia entre `WHERE` y `HAVING`.
- [ ] Dividir una consulta en pasos con CTE.
- [ ] Usar `SUM() OVER`, `LAG()` y `RANK()` con `PARTITION BY`.

**Próxima sesión (S26):** series de tiempo: índice temporal, `resample`, `rolling`, estacionalidad y pronóstico básico.